# EGM722 Assignment - Flood Exposure Analysis

Run `01_data_download.ipynb` first to download the data.

In [ ]:
import rasterio
import rasterio.merge
from rasterio.vrt import WarpedVRT
from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
from pathlib import Path
import glob


In [ ]:
# study area and settings
BBOX = (-96.5, 28.8, -94.5, 30.5) # west, south, east, north
FLOOD_THRESHOLD = 0.05 # backscatter below this = water
CRS = 'EPSG:32614' # UTM Zone 14N for Texas

raw_dir = Path('data/raw')
processed_dir = Path('data/processed')
outputs_dir = Path('outputs')
processed_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

## 1. Load the SAR data

In [ ]:
# find downloaded VV polarisation GeoTIFFs
vv_files = glob.glob(str(raw_dir / '**/*VV*.tif'), recursive=True)
vv_files += glob.glob(str(raw_dir / '*VV*.tif'))
print(f'Found {len(vv_files)} VV file(s)')

In [ ]:
# load all tiles and clip to the study area
datasets = [rasterio.open(f) for f in vv_files]
target_crs = datasets[0].crs

# OPERA tiles have different CRSs, WarpedVRT reprojects each tile so merge() 
# can combine them
vrts = []
for ds in datasets:
    if ds.crs != target_crs:
        vrts.append(WarpedVRT(ds, crs=target_crs))
    else:
        vrts.append(ds)

# rasterio.warp.transform_bounds converts our WGS84 BBOX into the raster CRS
# so merge() knows which portion of the tiles to read
clip_bounds = transform_bounds('EPSG:4326', target_crs, *BBOX)

# merge and clip
mosaic, mosaic_transform = rasterio.merge.merge(vrts, bounds=clip_bounds)
meta = datasets[0].meta.copy()
meta.update({'crs': target_crs, 'height': mosaic.shape[1],
             'width': mosaic.shape[2], 'transform': mosaic_transform})
for ds in datasets:
    ds.close()

clipped_path = processed_dir / 'sar_clipped.tif'
with rasterio.open(clipped_path, 'w', **meta) as dst:
    dst.write(mosaic)
print(f'SAR data loaded and clipped: {clipped_path}')


In [ ]:
# reproject to UTM Zone 14N (EPSG:32614) - the projected CRS for Texas
with rasterio.open(clipped_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, CRS, src.width, src.height, *src.bounds)
    meta = src.meta.copy()
    meta.update({'crs': CRS, 'transform': transform, 'width': width, 'height': height})
    reprojected_path = processed_dir / 'sar_utm.tif'
    with rasterio.open(reprojected_path, 'w', **meta) as dst:
        reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                  src_crs=src.crs, dst_crs=CRS, resampling=Resampling.bilinear)
print(f'Reprojected: {reprojected_path}')